# День 4 — Улучшение baseline

**Цель:** проверить несколько подходов на **том же** split и выбрать лучший.

Скриптовый аналог: [`train_improved.py`](../train_improved.py).

In [1]:
import sys, os
# Добавляем корень проекта в путь, чтобы импортировать модули (config, preprocess, ...)
sys.path.insert(0, os.path.abspath('..'))

In [2]:
import pandas as pd, joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score
from config import MODELS_DIR, RANDOM_STATE
from train_improved import load_split, WordCharVectorizer, LemmaTfidfVectorizer, _baseline_f1

In [3]:
df, train_idx, test_idx = load_split()
train_text, test_text = df.loc[train_idx, 'text_clean'], df.loc[test_idx, 'text_clean']
y_train = df.loc[train_idx, 'label'].to_numpy()
y_test = df.loc[test_idx, 'label'].to_numpy()
baseline_f1 = _baseline_f1()
print('Baseline macro F1:', round(baseline_f1, 4))

Baseline macro F1: 0.5987


## Сравнение подходов

In [4]:
results = [('Baseline: TF-IDF(1,2)+LogReg', baseline_f1)]
best = {'name': None, 'f1': -1, 'model': None, 'vec': None}

def evaluate(name, vec, model):
    Xtr, Xte = vec.fit_transform(train_text), vec.transform(test_text)
    model.fit(Xtr, y_train)
    f1 = f1_score(y_test, model.predict(Xte), average='macro')
    results.append((name, f1))
    if f1 > best['f1']: best.update(name=name, f1=f1, model=model, vec=vec)
    print(f'{name}: {f1:.4f}')

evaluate('TF-IDF(1,3)+LinearSVC', TfidfVectorizer(max_features=5000, ngram_range=(1,3)), LinearSVC(class_weight='balanced'))
evaluate('TF-IDF(1,2) sublinear+LogReg', TfidfVectorizer(max_features=5000, ngram_range=(1,2), sublinear_tf=True), LogisticRegression(max_iter=1000, class_weight='balanced'))
evaluate('Word+Char TF-IDF+LinearSVC', WordCharVectorizer(), LinearSVC(class_weight='balanced'))
evaluate('TF-IDF(1,2)+RandomForest', TfidfVectorizer(max_features=5000, ngram_range=(1,2)), RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=RANDOM_STATE))
# Лемматизация (spacy) может занять ~30-60 сек на первом запуске
evaluate('Lemmatization(spacy)+TF-IDF+LogReg', LemmaTfidfVectorizer(), LogisticRegression(max_iter=1000, class_weight='balanced'))

TF-IDF(1,3)+LinearSVC: 0.6646


TF-IDF(1,2) sublinear+LogReg: 0.6948


Word+Char TF-IDF+LinearSVC: 0.6800


TF-IDF(1,2)+RandomForest: 0.5609


Lemmatization(spacy)+TF-IDF+LogReg: 0.7147


## Подбор C для лучшей конфигурации (word+char + LinearSVC)

In [5]:
wc = WordCharVectorizer()
Xtr = wc.fit_transform(train_text)
grid = GridSearchCV(LinearSVC(class_weight='balanced'), {'C': [0.1, 0.5, 1, 2, 5]}, scoring='f1_macro', cv=3)
grid.fit(Xtr, y_train)
tuned = grid.best_estimator_
f1 = f1_score(y_test, tuned.predict(wc.transform(test_text)), average='macro')
name = f"Word+Char+LinearSVC (tuned C={grid.best_params_['C']})"
results.append((name, f1))
if f1 > best['f1']: best.update(name=name, f1=f1, model=tuned, vec=wc)
print(name, round(f1, 4))

Word+Char+LinearSVC (tuned C=0.5) 0.6983


## Итоговая таблица

In [6]:
tbl = pd.DataFrame(results, columns=['Метод', 'Macro F1'])
tbl['Улучшение'] = tbl['Macro F1'].apply(lambda f: '-' if abs(f - baseline_f1) < 1e-9 else f'{(f - baseline_f1) * 100:+.2f}%')
tbl

,Метод,Macro F1,Улучшение
0,"Baseline: TF-IDF(1,2)+LogReg",0.598700,-
1,"TF-IDF(1,3)+LinearSVC",0.664604,+6.59%
2,"TF-IDF(1,2) sublinear+LogReg",0.694841,+9.61%
3,Word+Char TF-IDF+LinearSVC,0.680020,+8.13%
4,"TF-IDF(1,2)+RandomForest",0.560907,-3.78%
5,Lemmatization(spacy)+TF-IDF+LogReg,0.714711,+11.60%
6,Word+Char+LinearSVC (tuned C=0.5),0.698326,+9.96%


In [7]:
joblib.dump(best['model'], MODELS_DIR / 'best_model.pkl')
joblib.dump(best['vec'], MODELS_DIR / 'best_vectorizer.pkl')
print('Лучший:', best['name'], round(best['f1'], 4))

Лучший: Lemmatization(spacy)+TF-IDF+LogReg 0.7147


**Вывод:** лучший вариант — **лемматизация (spacy) + TF-IDF + LogReg (balanced)**. Лемматизация схлопывает формы слов и снижает разреженность, а `class_weight='balanced'` поднимает recall редкого класса `negative`. RandomForest на разреженных TF-IDF уходит в majority-класс (`neutral`) и даёт худший результат.